In [ ]:
# Step 1: Install required packages...
!pip install -q gradio plotly pandas numpy vaderSentiment

import gradio as gr
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ==========================================
# 1. VISUAL ASSETS: LOGOS, THUMBNAILS & AVATARS
# ==========================================
PLATFORM_META = {
    'Instagram': {
        'logo': 'https://cdn-icons-png.flaticon.com/512/2111/2111463.png',
        'color': '#E1306C'
    },
    'TikTok': {
        'logo': 'https://cdn-icons-png.flaticon.com/512/3046/3046121.png',
        'color': '#00f2fe'
    },
    'LinkedIn': {
        'logo': 'https://cdn-icons-png.flaticon.com/512/3536/3536505.png',
        'color': '#0077B5'
    },
    'X (Twitter)': {
        'logo': 'https://cdn-icons-png.flaticon.com/512/5969/5969020.png',
        'color': '#1DA1F2'
    },
    'YouTube': {
        'logo': 'https://cdn-icons-png.flaticon.com/512/1384/1384060.png',
        'color': '#FF0000'
    }
}

# High-resolution post media thumbnails (Unsplash)
POST_THUMBNAILS = [
    'https://cdn.dribbble.com/userupload/23385917/file/original-658ad06201724410f3d5a0e07b926dd5.gif',
    'https://media.tenor.com/GZoBLzLNqL0AAAAm/x-twitter.webp',
    'https://static01.nyt.com/images/2019/03/07/style/07tiktok-1/07tiktok-still-1-superJumbo.gif',
    'https://images.unsplash.com/photo-1522202176988-66273c2fd55f?w=400&q=80',
    'https://images.unsplash.com/photo-1531482615713-2afd69097998?w=400&q=80',
    'https://images.unsplash.com/photo-1504868584819-f8e8b4b6d7e3?w=400&q=80'
]

# Influencer / Creator profile avatars
INFLUENCERS = [
    {'id': 'INF_01', 'name': 'Rahul Jaykar', 'tier': 'Macro', 'cost': 2500, 'avatar': 'https://images.unsplash.com/photo-1534528741775-53994a69daeb?w=200&q=80'},
    {'id': 'INF_02', 'name': 'Aarohi Keshav Shirke', 'tier': 'Micro', 'cost': 600, 'avatar': 'https://images.unsplash.com/photo-1517841905240-472988babdf9?w=200&q=80'},
    {'id': 'INF_03', 'name': 'Jordan JJ', 'tier': 'Mega', 'cost': 6000, 'avatar': 'https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=200&q=80'},
    {'id': 'INF_04', 'name': 'Heer Kaul', 'tier': 'Micro', 'cost': 750, 'avatar': 'https://images.unsplash.com/photo-1524504388940-b1c1722653e1?w=200&q=80'},
    {'id': 'INF_05', 'name': 'Radhe Mohan', 'tier': 'Mid-Tier', 'cost': 1500, 'avatar': 'https://images.unsplash.com/photo-1500648767791-00dcc994a43e?w=200&q=80'},
    {'id': 'INF_06', 'name': 'Nirjara Bharadwaj', 'tier': 'Organic', 'cost': 100, 'avatar': 'https://cdn-icons-png.flaticon.com/512/3135/3135715.png'}
]

# ==========================================
# 2. BACKEND DATA GENERATOR & NLP ENGINE
# ==========================================
def generate_visual_project_data(num_posts=1500):
    np.random.seed(42)
    random.seed(42)

    platforms = list(PLATFORM_META.keys())
    content_types = {
        'Instagram': ['Reel', 'Carousel', 'Static Image', 'Story'],
        'TikTok': ['Short Video', 'Live Stream', 'Photo Mode'],
        'LinkedIn': ['Article', 'Document/PDF', 'Text Post', 'Video'],
        'X (Twitter)': ['Thread', 'Short Post', 'Image Post', 'Video'],
        'YouTube': ['Long-form Video', 'YouTube Short', 'Community Post']
    }
    hashtags_pool = ['#AI', '#TechTrends', '#DigitalMarketing', '#GrowthHacking', '#StartupLife',
                     '#Innovation', '#FutureOfWork', '#DataScience', '#Branding', '#CreatorEconomy']
    comments_pool = [
        "Absolutely love this new feature! Game changer 🔥",
        "Best content on my feed today, keep it up!",
        "Super helpful breakdown, shared this with my team.",
        "Honestly, the recent update is terrible and buggy.",
        "Why is customer support ignoring my DMs? Disappointed.",
        "Way too expensive for what it actually offers.",
        "Interesting perspective, looking forward to part two.",
        "Incredible quality as always! Buying this right now 💯"
    ]

    analyzer = SentimentIntensityAnalyzer()
    start_date = datetime(2026, 1, 1)
    posts = []

    for i in range(1, num_posts + 1):
        plat = np.random.choice(platforms, p=[0.30, 0.25, 0.15, 0.20, 0.10])
        ctype = np.random.choice(content_types[plat])
        pdate = start_date + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))

        reach_mult = {'TikTok': 2.2, 'Instagram': 1.5, 'YouTube': 1.8, 'X (Twitter)': 1.0, 'LinkedIn': 0.7}[plat]
        reach = int(np.random.lognormal(mean=8.5, sigma=0.75) * reach_mult)
        impressions = int(reach * np.random.uniform(1.15, 1.65))

        eng_mod = 1.4 if any(v in ctype for v in ['Video', 'Reel', 'Short']) else 1.0
        likes = int(reach * np.random.uniform(0.02, 0.09) * eng_mod)
        comments = int(likes * np.random.uniform(0.05, 0.20))
        shares = int(likes * np.random.uniform(0.08, 0.25))
        saves = int(likes * np.random.uniform(0.10, 0.30))
        clicks = int(reach * np.random.uniform(0.005, 0.035))

        inf = random.choices(INFLUENCERS, weights=[0.05, 0.08, 0.03, 0.08, 0.06, 0.70])[0]
        conversions = int(clicks * np.random.uniform(0.02, 0.12) * (1.5 if inf['id'] != 'NONE' else 1.0))
        revenue = round(conversions * 49.99, 2)

        comment_txt = random.choice(comments_pool)
        score = analyzer.polarity_scores(comment_txt)['compound']
        sentiment = 'Positive' if score >= 0.05 else ('Negative' if score <= -0.05 else 'Neutral')

        posts.append({
            'post_id': f"POST_{i:04d}",
            'date': pd.to_datetime(pdate.date()),
            'hour': pdate.hour,
            'day_of_week': pdate.strftime('%A'),
            'platform': plat,
            'platform_logo': PLATFORM_META[plat]['logo'],
            'post_thumbnail': random.choice(POST_THUMBNAILS),
            'content_type': ctype,
            'hashtags': random.sample(hashtags_pool, k=random.randint(2, 4)),
            'influencer_name': inf['name'],
            'influencer_tier': inf['tier'],
            'influencer_avatar': inf['avatar'],
            'cost': inf['cost'],
            'impressions': impressions,
            'reach': reach,
            'likes': likes,
            'comments': comments,
            'shares': shares,
            'saves': saves,
            'clicks': clicks,
            'conversions': conversions,
            'revenue': revenue,
            'sentiment': sentiment
        })

    df = pd.DataFrame(posts)
    df['total_engagement'] = df['likes'] + df['comments'] + df['shares'] + df['saves']
    df['engagement_rate'] = round((df['total_engagement'] / df['reach']) * 100, 2)
    df['ctr'] = round((df['clicks'] / df['impressions']) * 100, 2)
    return df

df_master = generate_visual_project_data()
print("✅ Dataset & Image Assets Initialized!")

# ==========================================
# 3. DASHBOARD RENDERING ENGINE
# ==========================================
def update_visual_dashboard(selected_platforms, selected_tiers, sentiment_filter):
    if not selected_platforms:
        selected_platforms = list(PLATFORM_META.keys())
    if not selected_tiers:
        selected_tiers = df_master['influencer_tier'].unique().tolist()

    dff = df_master[
        (df_master['platform'].isin(selected_platforms)) &
        (df_master['influencer_tier'].isin(selected_tiers))
    ]
    if sentiment_filter != "All":
        dff = dff[dff['sentiment'] == sentiment_filter]

    if dff.empty:
        empty = go.Figure().update_layout(title="No data matches the selected filters.")
        return "No Data", "", "", empty, empty, empty, empty, empty, empty, empty, empty

    # --- A. CHANNEL CARDS WITH OFFICIAL LOGOS ---
    channel_cards_html = '<div style="display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 18px;">'
    for plat in selected_platforms:
        p_df = dff[dff['platform'] == plat]
        meta = PLATFORM_META[plat]
        p_reach = f"{p_df['reach'].sum():,}" if not p_df.empty else "0"
        p_er = f"{p_df['engagement_rate'].mean():.2f}%" if not p_df.empty else "0%"
        p_posts = len(p_df)

        channel_cards_html += f"""
        <div style="flex: 1; min-width: 175px; background: #1e293b; border-top: 4px solid {meta['color']};
                    padding: 14px; border-radius: 12px; color: white; display: flex; align-items: center; gap: 12px;
                    box-shadow: 0 4px 10px rgba(0,0,0,0.2);">
            <img src="{meta['logo']}" style="width: 46px; height: 46px; border-radius: 10px; background: white; padding: 5px; object-fit: contain;" />
            <div>
                <div style="font-weight: 700; font-size: 15px; color: #f8fafc;">{plat}</div>
                <div style="font-size: 12px; color: #94a3b8;">Reach: <b style="color:#38bdf8;">{p_reach}</b></div>
                <div style="font-size: 12px; color: #94a3b8;">Avg ER: <b style="color:#4ade80;">{p_er}</b> ({p_posts} posts)</div>
            </div>
        </div>
        """
    channel_cards_html += '</div>'

    # --- B. TOP PERFORMING POSTS GALLERY (WITH THUMBNAIL IMAGES & CHANNEL LOGOS) ---
    top_posts = dff.sort_values(by='engagement_rate', ascending=False).head(4)
    posts_gallery_html = '<div style="display: flex; flex-wrap: wrap; gap: 15px; margin-top: 10px;">'
    for _, row in top_posts.iterrows():
        posts_gallery_html += f"""
        <div style="flex: 1; min-width: 210px; background: #0f172a; border-radius: 12px; overflow: hidden; color: white; border: 1px solid #334155;">
            <div style="position: relative;">
                <img src="{row['post_thumbnail']}" style="width: 100%; height: 130px; object-fit: cover;" />
                <img src="{row['platform_logo']}" style="position: absolute; top: 8px; right: 8px; width: 28px; height: 28px; background: white; border-radius: 50%; padding: 3px;" />
                <span style="position: absolute; bottom: 8px; left: 8px; background: rgba(0,0,0,0.75); color: #facc15; padding: 3px 8px; border-radius: 6px; font-size: 11px; font-weight: bold;">
                    {row['content_type']}
                </span>
            </div>
            <div style="padding: 12px;">
                <div style="font-weight: bold; font-size: 13px; color: #e2e8f0;">{row['post_id']} • {row['platform']}</div>
                <div style="font-size: 12px; color: #94a3b8; margin-top: 4px;">
                    ❤️ {row['likes']:,} &nbsp;|&nbsp; 💬 {row['comments']:,} &nbsp;|&nbsp; 🔄 {row['shares']:,}
                </div>
                <div style="margin-top: 6px; font-size: 13px; color: #4ade80; font-weight: bold;">
                    🔥 Engagement Rate: {row['engagement_rate']}%
                </div>
            </div>
        </div>
        """
    posts_gallery_html += '</div>'

    # --- C. INFLUENCER PROFILE CARDS (WITH CREATOR AVATARS) ---
    inf_df = dff.groupby(['influencer_name', 'influencer_tier', 'influencer_avatar']).agg(
        spend=('cost', 'sum'), revenue=('revenue', 'sum'), eng=('total_engagement', 'sum'), posts=('post_id', 'count')
    ).reset_index()
    inf_df['ROI (%)'] = round(((inf_df['revenue'] - inf_df['spend']) / inf_df['spend']) * 100, 1)

    inf_cards_html = '<div style="display: flex; flex-wrap: wrap; gap: 14px; margin-top: 10px;">'
    for _, r in inf_df.sort_values(by='ROI (%)', ascending=False).iterrows():
        inf_cards_html += f"""
        <div style="flex: 1; min-width: 185px; background: #1e293b; padding: 14px; border-radius: 12px; color: white; text-align: center; border: 1px solid #334155;">
            <img src="{r['influencer_avatar']}" style="width: 64px; height: 64px; border-radius: 50%; object-fit: cover; border: 3px solid #38bdf8; margin: 0 auto 8px auto; display: block;" />
            <div style="font-weight: bold; font-size: 15px;">{r['influencer_name']}</div>
            <div style="font-size: 11px; color: #facc15; margin-bottom: 6px;">{r['influencer_tier']} Creator ({r['posts']} Posts)</div>
            <div style="font-size: 12px; color: #cbd5e1;">Revenue: <b style="color:#4ade80;">${r['revenue']:,.0f}</b></div>
            <div style="font-size: 13px; color: #f472b6; font-weight: bold; margin-top: 4px;">ROI: {r['ROI (%)']}%</div>
        </div>
        """
    inf_cards_html += '</div>'

    # --- D. INTERACTIVE CHARTS FOR ALL 8 SUBMODULES ---
    # 1. User Engagement
    eng_sums = dff[['likes', 'comments', 'shares', 'saves', 'clicks']].sum().reset_index()
    eng_sums.columns = ['Metric', 'Count']
    fig_eng = px.bar(
        eng_sums, x='Metric', y='Count', color='Metric',
        title="1. User Engagement Breakdown (Total Interactions)",
        template="plotly_white", text_auto='.2s'
    )

    # 2. Content Performance
    content_df = dff.groupby(['platform', 'content_type']).agg(
        reach=('reach', 'mean'), eng_rate=('engagement_rate', 'mean')
    ).reset_index().round(2)
    fig_content = px.treemap(
        content_df, path=[px.Constant("All Channels"), 'platform', 'content_type'],
        values='reach', color='eng_rate', color_continuous_scale='Viridis',
        title="2. Content Performance (Size = Avg Reach | Color = Avg Engagement Rate %)",
        template="plotly_white"
    )

    # 3. Hashtag Analytics
    df_tags = dff.explode('hashtags')
    tag_stats = df_tags.groupby('hashtags').agg(
        uses=('post_id', 'count'), avg_er=('engagement_rate', 'mean'), reach=('reach', 'mean'), conv=('conversions', 'sum')
    ).reset_index().round(2)
    fig_tags = px.scatter(
        tag_stats, x='uses', y='avg_er', size='reach', color='conv', text='hashtags',
        title="3. Hashtag Analytics (Usage vs. Engagement Rate & Conversions)",
        labels={'uses': 'Times Used', 'avg_er': 'Avg Engagement Rate (%)'},
        template="plotly_white", color_continuous_scale='Plasma'
    )
    fig_tags.update_traces(textposition='top center')

    # 4. Sentiment Analysis
    sent_df = dff.groupby(['platform', 'sentiment']).size().reset_index(name='posts')
    fig_sent = px.bar(
        sent_df, x='platform', y='posts', color='sentiment', barmode='group',
        color_discrete_map={'Positive': '#22c55e', 'Neutral': '#94a3b8', 'Negative': '#ef4444'},
        title="4. NLP Sentiment Analysis Across Channels",
        template="plotly_white"
    )

    # 5. Audience Analytics (Optimal Posting Heatmap)
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    heat_df = dff.pivot_table(index='day_of_week', columns='hour', values='engagement_rate', aggfunc='mean').reindex(days_order)
    fig_aud = px.imshow(
        heat_df, aspect="auto", color_continuous_scale="YlGnBu",
        title="5. Audience Analytics: Optimal Posting Time Heatmap (Engagement Rate %)",
        labels=dict(x="Hour of Day (24h)", y="Day of Week", color="Eng Rate %"),
        template="plotly_white"
    )

    # 6. Influencer Performance
    fig_inf = px.bar(
        inf_df, x='influencer_name', y=['spend', 'revenue'], barmode='group',
        title="6. Influencer Performance: Campaign Spend vs. Revenue ($)",
        template="plotly_white"
    )

    # 7. Engagement Trends
    trend_df = dff.groupby('date').agg(daily_eng=('total_engagement', 'sum')).reset_index().sort_values('date')
    trend_df['7d_ma'] = trend_df['daily_eng'].rolling(7, min_periods=1).mean()
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=trend_df['date'], y=trend_df['daily_eng'], mode='lines', name='Daily Engagement', opacity=0.35))
    fig_trend.add_trace(go.Scatter(x=trend_df['date'], y=trend_df['7d_ma'], mode='lines', name='7-Day Moving Avg', line=dict(color='#e11d48', width=3)))
    fig_trend.update_layout(title="7. Longitudinal Engagement Trends (Daily + 7-Day Moving Average)", template="plotly_white")

    # 8. Platform Comparison (With Embedded Channel Logos inside the Plotly Chart!)
    plat_df = dff.groupby('platform').agg(
        reach=('reach', 'sum'), eng_rate=('engagement_rate', 'mean'), ctr=('ctr', 'mean')
    ).reset_index().round(2)
    fig_plat = px.scatter(
        plat_df, x='eng_rate', y='ctr', size='reach', color='platform', text='platform',
        title="8. Platform Comparison (CTR % vs. Engagement Rate % — With Channel Logos)",
        labels={'eng_rate': 'Avg Engagement Rate (%)', 'ctr': 'Avg Click-Through Rate (%)'},
        template="plotly_white", size_max=55
    )
    fig_plat.update_traces(textposition='bottom center')

    # Embed channel logo icons directly on the scatter plot coordinates
    for _, r in plat_df.iterrows():
        fig_plat.add_layout_image(
            dict(
                source=PLATFORM_META[r['platform']]['logo'],
                x=r['eng_rate'], y=r['ctr'],
                xref="x", yref="y",
                sizex=0.35, sizey=0.12,
                xanchor="center", yanchor="middle",
                layer="above"
            )
        )

    return (
        channel_cards_html, posts_gallery_html, inf_cards_html,
        fig_eng, fig_content, fig_tags, fig_sent, fig_aud, fig_inf, fig_trend, fig_plat
    )

# ==========================================
# 4. BUILD & LAUNCH GRADIO WEB APPLICATION
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(), title="Visual Social Media Analytics Suite") as demo:
    gr.Markdown("# 📊 Visual Social Media Analytics Dashboard (All 8 Submodules)")
    gr.Markdown("Interactive Major Project — Complete with Channel Brand Logos, Post Media Thumbnails, Influencer Avatars, and Live Plotly Charts.")

    # Global Filter Bar
    with gr.Row():
        plat_filter = gr.CheckboxGroup(
            choices=list(PLATFORM_META.keys()),
            value=list(PLATFORM_META.keys()),
            label="🌐 Select Social Media Channels"
        )
        tier_filter = gr.CheckboxGroup(
            choices=['Organic', 'Micro', 'Mid-Tier', 'Macro', 'Mega'],
            value=['Organic', 'Micro', 'Mid-Tier', 'Macro', 'Mega'],
            label="🤝 Creator / Influencer Tier"
        )
        sent_filter = gr.Radio(
            choices=["All", "Positive", "Neutral", "Negative"],
            value="All",
            label="🧠 NLP Sentiment Filter"
        )

    # Top Channel Cards with Official Brand Logos
    gr.Markdown("### 🌐 Active Channels Overview")
    channel_banner = gr.HTML()

    # Organized Submodule Tabs
    with gr.Tabs():
        with gr.TabItem("1 & 2. Engagement & Visual Content"):
            gr.Markdown("#### 🖼️ Top Performing Posts (Visual Media Feed)")
            posts_gallery = gr.HTML()
            with gr.Row():
                plot_eng = gr.Plot()
                plot_content = gr.Plot()

        with gr.TabItem("3 & 4. Hashtags & NLP Sentiment"):
            with gr.Row():
                plot_tags = gr.Plot()
                plot_sent = gr.Plot()

        with gr.TabItem("5 & 6. Audience & Influencer Roster"):
            gr.Markdown("#### 🌟 Partnered Creators & Influencer Avatars")
            inf_gallery = gr.HTML()
            with gr.Row():
                plot_aud = gr.Plot()
                plot_inf = gr.Plot()

        with gr.TabItem("7 & 8. Trends & Platform Comparison"):
            with gr.Row():
                plot_trend = gr.Plot()
                plot_plat = gr.Plot()

    # Bind interactive callbacks
    inputs = [plat_filter, tier_filter, sent_filter]
    outputs = [
        channel_banner, posts_gallery, inf_gallery,
        plot_eng, plot_content, plot_tags, plot_sent, plot_aud, plot_inf, plot_trend, plot_plat
    ]

    for control in inputs:
        control.change(fn=update_visual_dashboard, inputs=inputs, outputs=outputs)

    demo.load(fn=update_visual_dashboard, inputs=inputs, outputs=outputs)

# Launch the app and create a public shareable link
demo.launch(share=True, debug=False)

✅ Dataset & Image Assets Initialized!


/tmp/ipykernel_915/1569939803.py:339: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://370d88d072c0015e6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
